In [ ]:
#practice set A

In [ ]:
#inner join
inner_df = orders_clean.join(customers_clean, "customer_id", "inner")
inner_df.show()


In [ ]:
#left join
left_df = orders_clean.join(customers_clean, "customer_id", "left")
left_df.show()


In [ ]:
#anti left
invalid_fk = orders_clean.join(customers_clean, "customer_id", "left_anti")
invalid_fk.show()


In [ ]:
## practice set2

In [ ]:
### total spend

total_spend = inner_df.groupBy("customer_id", "name", "city") \
    .agg(sum("amount").alias("total_spend"))
display(total_spend)


In [ ]:
#top 3 customers

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("city").orderBy(col("total_spend").desc())

top3 = total_spend.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 3)

top3.show()


In [ ]:
#Running total of sales

window_spec = Window.orderBy("order_date")

running_total = inner_df.withColumn(
    "running_total",
    sum("amount").over(window_spec)
)

running_total.show()


In [ ]:
#Rank customers by total spend

from pyspark.sql.functions import rank

ranked = total_spend.withColumn(
    "rank",
    rank().over(Window.orderBy(col("total_spend").desc()))
)

ranked.show()


In [ ]:
#Use LAG (previous order)

from pyspark.sql.functions import lag

window_spec = Window.partitionBy("customer_id").orderBy("order_date")

lag_df = inner_df.withColumn(
    "previous_aorder",
    lag("amount").over(window_spec)
)

lag_df.show()


In [ ]:
#Practice Set C: Date Analysis

In [ ]:
#Extract month

from pyspark.sql.functions import month

df_month = inner_df.withColumn("month", month("order_date"))
df_month.show()


In [ ]:
#Monthly sales aggregation

monthly_sales = df_month.groupBy("month") \
    .agg(sum("amount").alias("total_sales"))

monthly_sales.show()


In [ ]:
#Difference between dates

from pyspark.sql.functions import datediff

date_diff = inner_df.withColumn(
    "days_diff",
    datediff(col("order_date"), lag("order_date").over(Window.orderBy("order_date")))
)

date_diff.show()


In [ ]:
#Trend analysis by month

monthly_sales.orderBy("month").show()


In [ ]:
#Practice Set D: Timed Pipeline


In [ ]:
# Step 1: Clean data (already done)


In [ ]:
# Step 2: Validate FK
invalid_fk = orders_clean.join(customers_clean, "customer_id", "left_anti")


In [ ]:
# Step 3: Join
final_df = orders_clean.join(customers_clean, "customer_id", "inner")


In [ ]:
# Step 4: Aggregation
final_df = final_df.groupBy("customer_id", "name", "city") \
    .agg(
        sum("amount").alias("total_spend"),
        count("order_id").alias("order_count")
    )


In [ ]:
# Step 5: Window ranking
window_spec = Window.orderBy(col("total_spend").desc())

final_df = final_df.withColumn(
    "rank",
    rank().over(window_spec)
)


In [ ]:
# Step 6: Save output

In [ ]:
#final_df.write.mode("overwrite").csv("/tmp/phase6_output")

final_df.show()